# Function Tools - Give Agents Capabilities

## Overview
Learn how to extend agents with custom tools (functions) that they can call to perform specific actions.

## What You'll Learn
- Creating tools with the `@function_tool` decorator
- Adding tools to agents
- How agents decide when to use tools
- Best practices for tool descriptions

## Key Concepts
- **Tools**: Python functions that agents can call to retrieve information or perform actions
- **@function_tool**: Decorator that converts a Python function into an agent tool
- **Docstrings**: Tool descriptions that help the agent understand when to use them

## Step 1: Install Dependencies

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

In [ ]:
model_id = "openai.gpt-5.5"

## Step 2: Configure Authentication

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Step 3: Import Required Classes

Now we import `function_tool` to create tools.

In [ ]:
import asyncio

from agents import Agent, Runner, function_tool

## Step 4: Create a Function Tool

Use the `@function_tool` decorator to turn any Python function into a tool.

**Important:**
- The **docstring** becomes the tool description that the agent reads
- Write clear docstrings explaining what the tool does
- The agent uses this description to decide when to call the tool
- Add type hints for better validation

In [ ]:
@function_tool
def history_fun_fact() -> str:
    """Return a short history fact."""
    return "Sharks are older than trees."

## Step 5: Create Agent with Tools

Add tools to an agent using the `tools` parameter.

**Key points:**
- Pass tools as a list: `tools=[tool1, tool2, ...]`
- Mention tools in instructions to guide the agent
- The agent will automatically decide when to use tools based on user input

💡 **Tip**: Tell the agent WHEN to use tools in the instructions!

In [ ]:
agent = Agent(
    model=model_id,
    name="History tutor",
    instructions="Answer history questions clearly. Use history_fun_fact when it helps.",
    tools=[history_fun_fact],  # Add the tool here
)

## Step 6: Run the Agent

When you run the agent, it will:
1. Read your input
2. Decide if it needs to call any tools
3. Call the tool if appropriate
4. Use the tool's output to craft a response

🔍 **Watch**: The agent should call `history_fun_fact()` for this question!

In [ ]:
result = await Runner.run(agent, "Tell me something surprising about ancient life on Earth.")
print(result.final_output)

### Key Takeaways
- Tools extend agent capabilities beyond just conversation
- Clear docstrings help agents understand when to use tools
- Agents automatically decide when to call tools based on context
- Tools can have parameters and return any Python type